# SED at Critical Epochs of Galaxy Evolution

Given a list of **GroupIDs at z=0**, this notebook:

1. Traces progenitor merger trees back to the earliest available snapshot.
2. Builds continuous property histories (SFR, M★, M_mol) interpolated in cosmic time.
3. Identifies four **critical epochs** per galaxy:
   | Epoch | Definition |
   |---|---|
   | `sfr_peak` | Maximum of the interpolated SFR curve |
   | `quench_start` | sSFR first drops below 1/t (start of fading) |
   | `quench_end` | sSFR first drops below 0.2/t (fully quenched) |
   | `mol_depleted` | First downward crossing of M_mol/M★ = 10⁻⁴ |
4. Maps each epoch to the **nearest available snapshot**.
5. Extracts filtered particles and writes **Powderday master scripts** for every (galaxy, epoch) pair.

**Prerequisites:** progenitor FITS file at `output/<sim>/progenitors/progenitors_most_mass.fits`.

## 1 — Imports

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from astropy.cosmology import Planck15 as cosmo
from scipy.interpolate import PchipInterpolator

from simbanator.io.simba import Simulation
from simbanator.analysis.sfh_caesar import HDF5BuildHistory
from simbanator.analysis.quenching import find_quenching_times
from simbanator.analysis.particles import extract_particles
from simbanator.sed.makesed import MakeSED

plt.rcParams.update({'font.size': 13})

## 2 — Configuration

In [ ]:
# ── Simulation ────────────────────────────────────────────────────────────────
SIM_NAME    = 'cis100'
SNAP_Z0     = 149        # z = 0 snapshot
SNAP_OLDEST = 6          # oldest snapshot (deepest lookback)

# GroupIDs at the z=0 snapshot — edit this list
GROUND_IDS = [950, 990]

# ── Caesar HDF5 property keys ─────────────────────────────────────────────────
SFR_KEY       = 'sfr'             # SFR [Msun/yr]
STAR_MASS_KEY = 'masses.stellar'  # M★ [Msun]
MOL_MASS_KEY  = 'masses.H2'       # molecular (H2) mass [Msun]

# ── Critical epoch thresholds ─────────────────────────────────────────────────
MOL_THRESH = 1e-4    # first downward crossing of M_mol / M★ below this value

# ── Quenching finder knobs ────────────────────────────────────────────────────
QUENCH_SMOOTH   = 3     # uniform_filter1d window on log-sSFR before interpolation
QUENCH_INTERP   = 20    # interpolation density factor
QUENCH_TOL      = 0.1   # dex tolerance for the 1/t persistence check
QUENCH_POST     = 0.2   # persistence fraction (end_t = qt + post * qt)

# ── Particle extraction ───────────────────────────────────────────────────────
SED_PREFIX          = 'm100n1024'
OVERWRITE_PARTICLES = False   # True to re-extract already-existing files

# ── Powderday / SED ───────────────────────────────────────────────────────────
SED_RUN_TAG         = 'critical_epochs_dust_on'
SED_RUN_TAG_OFF     = 'critical_epochs_dust_off'
SED_PARAM_FILE      = 'parameters_master.py'
SED_PARAM_FILE_OFF  = 'parameters_master-nodust.py'
SED_PARTITION       = 'INTEL_PHI'
SELECTION_FILE      = 'selection_critical_epochs'

# ── Remote output path (GVFS tunnel to cluster) ───────────────────────────────
GVFS_BASE   = "/run/user/1000/gvfs/sftp:host=slurmusrint2.cis.gov.pl,user=glorenzon"
REMOTE_HOME = os.path.join(GVFS_BASE, "mnt/home/glorenzon/analize_simba_cgm")
OUTPUT_SED  = os.path.join(REMOTE_HOME, 'output', SIM_NAME, 'sed')

# ── Epoch catalogue ───────────────────────────────────────────────────────────
EPOCH_NAMES = ['sfr_peak', 'quench_start', 'quench_end', 'mol_depleted']
EPOCH_LABELS = {
    'sfr_peak':     'SFR peak',
    'quench_start': 'Quench start (sSFR < 1/t)',
    'quench_end':   'Quench end (sSFR < 0.2/t)',
    'mol_depleted': r'$M_\mathrm{mol}/M_\star < 10^{-4}$',
}
EPOCH_COLORS = {
    'sfr_peak':     'tab:blue',
    'quench_start': 'tab:orange',
    'quench_end':   'tab:red',
    'mol_depleted': 'tab:purple',
}

## 3 — Load simulation handle and z=0 Caesar catalog

In [ ]:
sim   = Simulation(SIM_NAME)
cs_z0 = sim.load_catalog(SNAP_Z0)

z_z0  = sim.get_z_from_snap(SNAP_Z0)
print(f"Simulation : {SIM_NAME}")
print(f"z=0 snap   : {SNAP_Z0}  (z = {z_z0:.4f})")
print(f"Galaxies at z=0 in catalog: {len(cs_z0.galaxies)}")
print(f"Input GroupIDs: {GROUND_IDS}")

## 4 — Build progenitor chains

In [ ]:
hist = HDF5BuildHistory(sim, cs_z0)
hist.get_history_indx(GROUND_IDS, start_snap=SNAP_Z0, end_snap=SNAP_OLDEST)

n_snaps = len(hist.history_indx)
print(f"Progenitor chains built over {n_snaps} snapshots ({SNAP_OLDEST}–{SNAP_Z0}).")

## 5 — Retrieve property histories

In [ ]:
propr_request = {'galaxy_data': [SFR_KEY, STAR_MASS_KEY, MOL_MASS_KEY]}

z_dict, props = hist.get_property_history(propr_request, verbose=1)

# Arrays have shape (n_snaps, n_gals); snapshot order is decreasing (SNAP_Z0 → SNAP_OLDEST)
print(f"\nLoaded properties: {list(props.keys())}")
print(f"Shape: {props[SFR_KEY].shape}  (n_snaps × n_gals)")

## 6 — Find critical epochs

All times are derived from PCHIP-interpolated curves in cosmic time.  
The nearest snapshot is identified in the next section.

In [ ]:
# ── Shared time axis (ascending cosmic time) ──────────────────────────────────
zs        = z_dict['Redshift']          # decreasing redshift → increasing time
snaps_arr = z_dict['Snapshot'].astype(int)
t_yr      = cosmo.age(zs).value * 1e9  # cosmic time [yr]

sort_idx     = np.argsort(t_yr)         # oldest → youngest
t_sorted     = t_yr[sort_idx]
snaps_sorted = snaps_arr[sort_idx]

# ── Helper: first downward crossing of a curve through a threshold ─────────────
def _first_downward_crossing(t_dense, y_dense, threshold):
    """Return interpolated time of first downward crossing, or NaN."""
    d = y_dense - threshold
    for i in range(len(d) - 1):
        if d[i] > 0 and d[i + 1] <= 0:
            frac = d[i] / (d[i] - d[i + 1])
            return t_dense[i] + frac * (t_dense[i + 1] - t_dense[i])
    return np.nan


critical_times = {}   # gid → {epoch: t_yr or NaN}

for j, gid in enumerate(GROUND_IDS):
    print(f"\n── Galaxy {gid} ──────────────────────────────")

    # Property arrays for this galaxy, sorted by ascending cosmic time
    sfr_j   = props[SFR_KEY][:, j][sort_idx]
    mstar_j = props[STAR_MASS_KEY][:, j][sort_idx]
    mmol_j  = props[MOL_MASS_KEY][:, j][sort_idx]

    # Keep only finite, physically meaningful snapshots
    valid = (
        np.isfinite(sfr_j) & np.isfinite(mstar_j) &
        (mstar_j > 0) & np.isfinite(t_sorted)
    )
    t_v     = t_sorted[valid]
    sfr_v   = sfr_j[valid]
    mstar_v = mstar_j[valid]
    mmol_v  = mmol_j[valid]

    n_valid = np.sum(valid)
    print(f"  Valid snapshots: {n_valid} / {len(t_sorted)}")

    if n_valid < 4:
        warnings.warn(f"Too few valid snapshots for galaxy {gid} — setting all epochs to NaN.")
        critical_times[gid] = {name: np.nan for name in EPOCH_NAMES}
        continue

    # Dense interpolation grid (PCHIP preserves monotonicity)
    n_dense = n_valid * QUENCH_INTERP
    t_dense = np.linspace(t_v[0], t_v[-1], n_dense)

    # ── 1. Peak SFR ────────────────────────────────────────────────────────────
    sfr_interp = PchipInterpolator(t_v, np.clip(sfr_v, 1e-14, None))
    sfr_dense  = sfr_interp(t_dense)
    t_sfr_peak = float(t_dense[np.argmax(sfr_dense)])
    print(f"  SFR peak   : t = {t_sfr_peak/1e9:.3f} Gyr")

    # ── 2. Quenching start + end (via find_quenching_times) ────────────────────
    ssfr_v = sfr_v / mstar_v
    qt_list, sft_list, _ = find_quenching_times(
        t_v, ssfr_v,
        galaxy_id=gid,
        smooth_window=QUENCH_SMOOTH,
        interp_factor=QUENCH_INTERP,
        tolerance=QUENCH_TOL,
        post_quench_frac=QUENCH_POST,
        save_fits_path=None,
        plot=False,
    )
    if qt_list:
        t_sft = float(sft_list[0])   # start of fading → quench start
        t_qt  = float(qt_list[0])    # full quench     → quench end
        print(f"  Quench start: t = {t_sft/1e9:.3f} Gyr")
        print(f"  Quench end  : t = {t_qt/1e9:.3f} Gyr")
        if len(qt_list) > 1:
            print(f"  ({len(qt_list)} quenching events found; using first)")
    else:
        t_sft = np.nan
        t_qt  = np.nan
        print("  No quenching event detected.")

    # ── 3. M_mol / M★ < MOL_THRESH first downward crossing ────────────────────
    valid_mol = np.isfinite(mmol_v) & (mmol_v >= 0)
    t_mol_cross = np.nan
    if np.sum(valid_mol) >= 2:
        ratio      = mmol_v[valid_mol] / mstar_v[valid_mol]
        t_mol_v    = t_v[valid_mol]
        ratio_interp = PchipInterpolator(t_mol_v, ratio)
        t_mol_dense  = np.linspace(t_mol_v[0], t_mol_v[-1], len(t_mol_v) * QUENCH_INTERP)
        ratio_dense  = ratio_interp(t_mol_dense)
        t_mol_cross  = _first_downward_crossing(t_mol_dense, ratio_dense, MOL_THRESH)
        if np.isfinite(t_mol_cross):
            print(f"  Mol depleted: t = {t_mol_cross/1e9:.3f} Gyr")
        else:
            print("  M_mol/M★ never crosses threshold (always below or always above).")
    else:
        print("  Insufficient M_mol data.")

    critical_times[gid] = {
        'sfr_peak':     t_sfr_peak,
        'quench_start': t_sft,
        'quench_end':   t_qt,
        'mol_depleted': t_mol_cross,
    }

## 7 — Map critical epochs to nearest snapshot

In [ ]:
def _nearest_snap(t_target, snaps, t_snaps):
    """Return the snapshot number whose cosmic time is closest to t_target."""
    if not np.isfinite(t_target):
        return None
    return int(snaps[np.argmin(np.abs(t_snaps - t_target))])


critical_snaps = {}   # gid → {epoch: snap_number or None}

for gid in GROUND_IDS:
    critical_snaps[gid] = {
        epoch: _nearest_snap(t, snaps_sorted, t_sorted)
        for epoch, t in critical_times[gid].items()
    }

print(f"{'GID':>8}  {'Epoch':<25}  {'t [Gyr]':>10}  {'z':>7}  {'Snap':>6}")
print("-" * 65)
for gid in GROUND_IDS:
    for epoch in EPOCH_NAMES:
        t   = critical_times[gid][epoch]
        sn  = critical_snaps[gid][epoch]
        if sn is not None:
            z_ep = sim.get_z_from_snap(sn)
            print(f"{gid:>8}  {EPOCH_LABELS[epoch]:<25}  {t/1e9:>10.3f}  {z_ep:>7.4f}  {sn:>6}")
        else:
            print(f"{gid:>8}  {EPOCH_LABELS[epoch]:<25}  {'N/A':>10}  {'—':>7}  {'—':>6}")

## 8 — Progenitor IDs at critical snapshots

In [ ]:
def _prog_id(hist_obj, gal_j_idx, snap_target):
    """Return the progenitor GroupID (= HDF5 row index) at snap_target, or None."""
    snap_str = str(snap_target)
    if snap_str not in hist_obj.history_indx:
        return None
    val = hist_obj.history_indx[snap_str][gal_j_idx]
    return None if np.isnan(val) else int(val)


# For each galaxy × epoch, store the progenitor ID at the target snapshot
critical_prog_ids = {}   # (gid, epoch) → prog_id or None

print(f"{'GID':>8}  {'Epoch':<25}  {'Snap':>6}  {'ProgID':>8}")
print("-" * 55)
for j, gid in enumerate(GROUND_IDS):
    for epoch in EPOCH_NAMES:
        sn = critical_snaps[gid][epoch]
        if sn is None:
            pid = None
        else:
            pid = _prog_id(hist, j, sn)
        critical_prog_ids[(gid, epoch)] = pid
        pid_str = str(pid) if pid is not None else 'N/A'
        snap_str = str(sn) if sn is not None else '—'
        print(f"{gid:>8}  {EPOCH_LABELS[epoch]:<25}  {snap_str:>6}  {pid_str:>8}")

## 9 — Diagnostic plots

History curves with critical epochs marked — use these to sanity-check epoch detection.

In [ ]:
for j, gid in enumerate(GROUND_IDS):

    sfr_j   = props[SFR_KEY][:, j][sort_idx]
    mstar_j = props[STAR_MASS_KEY][:, j][sort_idx]
    mmol_j  = props[MOL_MASS_KEY][:, j][sort_idx]
    t_gyr   = t_sorted / 1e9

    valid   = np.isfinite(sfr_j) & np.isfinite(mstar_j) & (mstar_j > 0)
    t_v     = t_sorted[valid]
    sfr_v   = sfr_j[valid]
    mstar_v = mstar_j[valid]
    mmol_v  = mmol_j[valid]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'Galaxy {gid} — critical epoch detection', fontweight='bold')

    # Panel 1: SFR
    ax = axes[0]
    ax.semilogy(t_v / 1e9, np.clip(sfr_v, 1e-4, None), 'o-', lw=1.5, ms=3, color='steelblue')
    t_pk = critical_times[gid]['sfr_peak']
    if np.isfinite(t_pk):
        ax.axvline(t_pk / 1e9, color=EPOCH_COLORS['sfr_peak'], lw=2, ls='--',
                   label=f"SFR peak ({t_pk/1e9:.2f} Gyr)")
    ax.set_xlabel('Cosmic time [Gyr]')
    ax.set_ylabel(r'SFR [$M_\odot\,\mathrm{yr}^{-1}$]')
    ax.set_title('Star formation rate')
    ax.legend(fontsize=9)

    # Panel 2: sSFR with 1/t and 0.2/t thresholds
    ax = axes[1]
    ssfr_v = sfr_v / mstar_v
    ax.semilogy(t_v / 1e9, ssfr_v, 'o-', lw=1.5, ms=3, color='steelblue', label='sSFR')
    t_lin = np.linspace(t_v[0], t_v[-1], 200)
    ax.semilogy(t_lin / 1e9, 1 / t_lin, 'k--', lw=1.2, label=r'$1/t$')
    ax.semilogy(t_lin / 1e9, 0.2 / t_lin, 'k:', lw=1.2, label=r'$0.2/t$')
    for ep in ('quench_start', 'quench_end'):
        t_ep = critical_times[gid][ep]
        if np.isfinite(t_ep):
            ax.axvline(t_ep / 1e9, color=EPOCH_COLORS[ep], lw=2, ls='--',
                       label=f"{ep.replace('_', ' ')} ({t_ep/1e9:.2f} Gyr)")
    ax.set_xlabel('Cosmic time [Gyr]')
    ax.set_ylabel(r'sSFR [$\mathrm{yr}^{-1}$]')
    ax.set_title('Specific SFR — quenching')
    ax.legend(fontsize=9)

    # Panel 3: M_mol / M★
    ax = axes[2]
    valid_mol = np.isfinite(mmol_v) & (mmol_v >= 0)
    if np.sum(valid_mol) >= 2:
        ax.semilogy(t_v[valid_mol] / 1e9, mmol_v[valid_mol] / mstar_v[valid_mol],
                    'o-', lw=1.5, ms=3, color='steelblue', label=r'$M_\mathrm{mol}/M_\star$')
    ax.axhline(MOL_THRESH, color='gray', lw=1.5, ls='--',
               label=f'threshold = {MOL_THRESH:.0e}')
    t_mol = critical_times[gid]['mol_depleted']
    if np.isfinite(t_mol):
        ax.axvline(t_mol / 1e9, color=EPOCH_COLORS['mol_depleted'], lw=2, ls='--',
                   label=f'depleted ({t_mol/1e9:.2f} Gyr)')
    ax.set_xlabel('Cosmic time [Gyr]')
    ax.set_ylabel(r'$M_\mathrm{mol}\,/\,M_\star$')
    ax.set_title('Molecular gas fraction')
    ax.legend(fontsize=9)

    plt.tight_layout()
    out_dir = os.path.join(os.getcwd(), 'output', 'figures', 'critical_epochs')
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, f'gal{gid}_critical_epochs.pdf'), bbox_inches='tight')
    plt.show()

## 10 — Collect unique (snapshot, progenitor ID) extraction targets

In [ ]:
# extraction_targets: (snap, prog_id) → [(ground_id, epoch), ...]
extraction_targets = {}

for gid in GROUND_IDS:
    for epoch in EPOCH_NAMES:
        sn  = critical_snaps[gid][epoch]
        pid = critical_prog_ids[(gid, epoch)]
        if sn is None or pid is None:
            continue
        key = (sn, pid)
        extraction_targets.setdefault(key, []).append((gid, epoch))

print(f"Unique (snap, progenitor_id) pairs to process: {len(extraction_targets)}\n")
print(f"{'Snap':>6}  {'ProgID':>8}  {'z':>7}  Galaxies × epochs")
print("-" * 60)
for (sn, pid), tags in sorted(extraction_targets.items()):
    z_ep = sim.get_z_from_snap(sn)
    tag_str = ', '.join(f"{g}@{ep}" for g, ep in tags)
    print(f"{sn:>6}  {pid:>8}  {z_ep:>7.4f}  {tag_str}")

## 11 — Extract filtered particles

Groups extractions by snapshot to load each Caesar catalog only once.

In [ ]:
from collections import defaultdict

# Group by snapshot
by_snap = defaultdict(list)   # snap → [prog_id, ...]
for (sn, pid) in extraction_targets:
    by_snap[sn].append(pid)

extracted_files = {}   # (snap, prog_id) → output path

for sn in sorted(by_snap.keys()):
    gal_ids_at_snap = by_snap[sn]
    print(f"\nsnap {sn:03d}  (z = {sim.get_z_from_snap(sn):.4f})  —  {len(gal_ids_at_snap)} galaxy/galaxies")

    cs_snap  = sim.load_catalog(sn)
    simfile  = sim.get_snapshot_file(sn)

    paths = extract_particles(
        cs_snap,
        simfile,
        snap=sn,
        galaxy_ids=gal_ids_at_snap,
        sim_name=sim.name,
        overwrite=OVERWRITE_PARTICLES,
        prefix=SED_PREFIX,
    )

    if paths is not None:
        for pid, path in zip(gal_ids_at_snap, paths if hasattr(paths, '__iter__') else [paths]):
            extracted_files[(sn, pid)] = path

print("\nParticle extraction complete.")

## 12 — Write Powderday selection and master scripts

In [ ]:
hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered')

# Ordered list of (snap, prog_id) pairs — shared by both runs
target_snaps   = [sn  for sn, pid in sorted(extraction_targets.keys())]
target_gal_ids = [pid for sn, pid in sorted(extraction_targets.keys())]

# ── Dust-on run ───────────────────────────────────────────────────────────────
makesed_on = MakeSED(
    sim,
    nnodes=1,
    model_run_name=SED_RUN_TAG,
    hydro_dir_base=hydro_dir_base,
    selection_file=SELECTION_FILE,
    output_dir=OUTPUT_SED,
    run_tag=SED_RUN_TAG,
)
makesed_on.selection_gals(snaps=target_snaps, galaxyID=target_gal_ids)
makesed_on.create_master(
    where='cluster',
    subset_type='plist',
    radius=None,
    partition=SED_PARTITION,
    prefix=SED_PREFIX,
    paramf=SED_PARAM_FILE,
    snaps_to_run=None,
)
print("dust_on master scripts written.")

# ── Dust-off run ──────────────────────────────────────────────────────────────
makesed_off = MakeSED(
    sim,
    nnodes=1,
    model_run_name=SED_RUN_TAG_OFF,
    hydro_dir_base=hydro_dir_base,
    selection_file=SELECTION_FILE,
    output_dir=OUTPUT_SED,
    run_tag=SED_RUN_TAG_OFF,
)
makesed_off.selection_gals(snaps=target_snaps, galaxyID=target_gal_ids)
makesed_off.create_master(
    where='cluster',
    subset_type='plist',
    radius=None,
    partition=SED_PARTITION,
    prefix=SED_PREFIX,
    paramf=SED_PARAM_FILE_OFF,
    snaps_to_run=None,
)
print("dust_off master scripts written.")

## 13 — Final summary

In [ ]:
print("=" * 80)
print(f"SED at critical epochs — {SIM_NAME}  run tag: {SED_RUN_TAG}")
print("=" * 80)
print(f"{'GID':>8}  {'Epoch':<25}  {'t [Gyr]':>10}  {'z':>7}  {'Snap':>6}  {'ProgID':>8}")
print("-" * 80)
for gid in GROUND_IDS:
    for epoch in EPOCH_NAMES:
        t   = critical_times[gid][epoch]
        sn  = critical_snaps[gid][epoch]
        pid = critical_prog_ids[(gid, epoch)]
        t_str   = f"{t/1e9:>10.3f}" if np.isfinite(t)  else f"{'N/A':>10}"
        z_str   = f"{sim.get_z_from_snap(sn):>7.4f}" if sn is not None else f"{'—':>7}"
        sn_str  = f"{sn:>6}" if sn  is not None else f"{'—':>6}"
        pid_str = f"{pid:>8}" if pid is not None else f"{'—':>8}"
        print(f"{gid:>8}  {EPOCH_LABELS[epoch]:<25}  {t_str}  {z_str}  {sn_str}  {pid_str}")
    print()

print(f"\n{len(extraction_targets)} unique (snap, galaxy) Powderday jobs prepared.")
model_dir = os.path.join(OUTPUT_SED, SED_RUN_TAG, 'powderday_sed_out')
submit_script = os.path.join(model_dir, 'submit_all_snaps.sh')
print(f"\nTo run on the cluster:")
print(f"  bash {submit_script}")

## 14 — SED figures at critical epochs

One figure per galaxy: all critical-epoch SEDs overlaid on a single log–log plot.
Solid lines = dust-on run; dashed lines = dust-off run (toggle with `PLOT_DUST_OFF`).
Wavelengths default to rest-frame for physical comparison across epochs at different *z*.

In [ ]:
from hyperion.model import ModelOutput
from astropy import units as u, constants
from astropy.cosmology import Planck13
from matplotlib.lines import Line2D

# ── Options ───────────────────────────────────────────────────────────────────
PLOT_INCLINATION = 0       # SED inclination index (0 = first available)
PLOT_REST_FRAME  = True    # True → rest-frame λ; False → observed-frame λ
PLOT_DUST_OFF    = True    # also overlay dust-off SED (dashed lines)

# ── Helper ────────────────────────────────────────────────────────────────────
def _load_sed_mjy(sed_path, z, rest_frame=True):
    """Load a Powderday .rtout.sed; return (wav [µm Quantity], flux [mJy, shape (n_incl, n_wav)])."""
    m = ModelOutput(sed_path)
    wav_raw, flux_raw = m.get_sed(inclination='all', aperture=-1)
    wav  = np.asarray(wav_raw) * u.micron
    if not rest_frame:
        wav = wav * (1. + z)
    flux = np.asarray(flux_raw) * u.erg / u.s
    dl   = Planck13.luminosity_distance(z).to(u.cm)
    flux /= (4. * np.pi * dl ** 2)
    nu   = (constants.c.cgs / wav.to(u.cm)).to(u.Hz)
    return wav, (flux / nu).to(u.mJy)

# ── Output directory ──────────────────────────────────────────────────────────
fig_dir = os.path.join(os.getcwd(), 'output', 'figures', 'sed_critical_epochs')
os.makedirs(fig_dir, exist_ok=True)

# ── One figure per galaxy ─────────────────────────────────────────────────────
for gid in GROUND_IDS:
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.suptitle(f'Galaxy {gid} — SED at critical epochs', fontweight='bold')

    epoch_handles = []
    any_plotted   = False

    for epoch in EPOCH_NAMES:
        sn  = critical_snaps[gid][epoch]
        pid = critical_prog_ids[(gid, epoch)]
        if sn is None or pid is None:
            continue

        z     = sim.get_z_from_snap(sn)
        color = EPOCH_COLORS[epoch]
        label = f"{EPOCH_LABELS[epoch]}  (z = {z:.3f})"

        # dust-on SED (solid line)
        path_on = os.path.join(
            makesed_on.model_dir_base,
            f'snap_{sn:03}', f'gal_{pid}',
            f'snap{sn:03}.galaxy{pid:06}.rtout.sed',
        )
        if os.path.isfile(path_on):
            wav, flux = _load_sed_mjy(path_on, z, rest_frame=PLOT_REST_FRAME)
            ax.loglog(wav.value, flux[PLOT_INCLINATION].value,
                      color=color, lw=2.0, zorder=3)
            epoch_handles.append(Line2D([0], [0], color=color, lw=2.0, label=label))
            any_plotted = True
        else:
            warnings.warn(f"Gal {gid}, epoch {epoch}: dust-on SED not found — {path_on}")

        # dust-off SED (dashed line, optional)
        if PLOT_DUST_OFF:
            path_off = os.path.join(
                makesed_off.model_dir_base,
                f'snap_{sn:03}', f'gal_{pid}',
                f'snap{sn:03}.galaxy{pid:06}.rtout.sed',
            )
            if os.path.isfile(path_off):
                wav, flux = _load_sed_mjy(path_off, z, rest_frame=PLOT_REST_FRAME)
                ax.loglog(wav.value, flux[PLOT_INCLINATION].value,
                          color=color, lw=1.2, ls='--', alpha=0.55, zorder=2)

    if not any_plotted:
        print(f"Galaxy {gid}: no SED files found — figure skipped.")
        plt.close(fig)
        continue

    xlabel = (r'$\lambda_\mathrm{rest}\;[\mu\mathrm{m}]$'
              if PLOT_REST_FRAME else r'$\lambda_\mathrm{obs}\;[\mu\mathrm{m}]$')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Flux density (mJy)')
    ax.set_xlim(0.05, 1500)

    legend_epochs = ax.legend(handles=epoch_handles, fontsize=9,
                               loc='upper left', title='Epoch')
    ax.add_artist(legend_epochs)

    if PLOT_DUST_OFF:
        style_handles = [
            Line2D([0], [0], color='k', lw=2.0,                          label='dust on'),
            Line2D([0], [0], color='k', lw=1.2, ls='--', alpha=0.6,      label='dust off'),
        ]
        ax.legend(handles=style_handles, fontsize=9, loc='lower right')

    plt.tight_layout()
    outpath = os.path.join(fig_dir, f'gal{gid}_sed_epochs.pdf')
    fig.savefig(outpath, bbox_inches='tight')
    print(f"Saved → {outpath}")
    plt.show()